In [1]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("explore").getOrCreate()

df = spark.read.parquet("E:/Python/DataEngineer/MiniProject/nyc_taxi_platform/data/bronze/year=2024/month=01/yellow_tripdata_2024-01.parquet")

In [2]:
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



In [3]:
df.show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:57:55|  2024-01-01 01:17:43|              1|         1.72|         1|                 N|         186|          79|           2|       17.7|  1.0|    0.5|       0.

In [4]:
df.filter(df.trip_distance > 10).show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       2| 2024-01-01 00:49:44|  2024-01-01 01:15:47|              2|        10.82|         1|                 N|         138|         181|           1|       45.7|  6.0|    0.5|      10.

In [5]:
df.select("fare_amount", "tip_amount", "total_amount")\
    .summary("min", "max", "count")\
    .show()

+-------+-----------+----------+------------+
|summary|fare_amount|tip_amount|total_amount|
+-------+-----------+----------+------------+
|    min|     -899.0|     -80.0|      -900.0|
|    max|     5000.0|     428.0|      5000.0|
|  count|    2964624|   2964624|     2964624|
+-------+-----------+----------+------------+



In [6]:
df.select("trip_distance").summary("min", "25%", "50%", "75%", "max").show()
print(f"Số chuyến đi có khoảng cách = 0: f{df.filter(df.trip_distance <= 0).count()}")

+-------+-------------+
|summary|trip_distance|
+-------+-------------+
|    min|          0.0|
|    25%|          1.0|
|    50%|         1.68|
|    75%|         3.11|
|    max|     312722.3|
+-------+-------------+

Số chuyến đi có khoảng cách = 0: f60371


In [7]:
df.groupBy("passenger_count").count().orderBy("passenger_count").show()

+---------------+-------+
|passenger_count|  count|
+---------------+-------+
|           NULL| 140162|
|              0|  31465|
|              1|2188739|
|              2| 405103|
|              3|  91262|
|              4|  51974|
|              5|  33506|
|              6|  22353|
|              7|      8|
|              8|     51|
|              9|      1|
+---------------+-------+



In [9]:
invalid_locations = df.filter((df.PULocationID < 1) | (df.PULocationID > 265) |
                              (df.DOLocationID < 1) | (df.DOLocationID > 265))
print(f"Số chuyến đi có mã vùng không hợp lệ: {invalid_locations.count()}")

Số chuyến đi có mã vùng không hợp lệ: 0


In [10]:
from pyspark.sql.functions import col, count, when
df.select([
    count(
        when(col(c).isNull(), c)
    ).alias(c)
    for c in df.columns
]).show()


+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       0|                   0|                    0|         140162|            0|    140162|            140162|           0|           0|           0|          0|    0|      0|         

In [1]:
import os
import sys
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# 1. Cấu hình HADOOP_HOME cho môi trường Windows local
project_root = "E:/Python/DataEngineer/MiniProject/nyc_taxi_platform"
hadoop_home = os.path.join(project_root, "hadoop")
os.environ["HADOOP_HOME"] = hadoop_home
os.environ["PATH"] += os.pathsep + os.path.join(hadoop_home, "bin")

# 2. Khởi tạo Spark Session có cấu hình RAM 8GB và Java 21
spark = (SparkSession.builder
    .appName("explore")
    .config("spark.driver.memory", "8g")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.driver.extraJavaOptions", "--add-opens=java.base/sun.nio.ch=ALL-UNNAMED --add-opens=java.base/java.nio=ALL-UNNAMED")
    .getOrCreate())

print("Spark Session trong Notebook đã khởi tạo thành công!")


# 1. Đọc dữ liệu từ Silver
df_silver = spark.read.parquet("E:/Python/DataEngineer/MiniProject/nyc_taxi_platform/data/silver")

# 2. Đọc bảng taxi_zone_lookup
zone_df = spark.read.option("header", "true").csv("E:/Python/DataEngineer/MiniProject/nyc_taxi_platform/data/taxi_zone_lookup.csv")

# 3. Join dữ liệu
df_joined = df_silver.join(
    zone_df.alias("pickup_zone"),
    df_silver.PULocationID == F.col("pickup_zone.LocationID"),
    "left"
).withColumnRenamed("Zone", "pickup_zone_name") \
 .withColumnRenamed("Borough", "pickup_borough") \
 .drop("LocationID")

# 4. Lấy 5 dòng và chuyển sang Pandas để hiển thị dạng bảng HTML đẹp mắt
df_joined.limit(5).toPandas()


Spark Session trong Notebook đã khởi tạo thành công!


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,...,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,year,month,pickup_borough,pickup_zone_name,service_zone
0,1,2025-10-01 00:15:32,2025-10-01 01:04:03,1,17.20,2,N,132,107,1,...,6.94,1.0,83.44,2.5,1.75,2025,10,Queens,JFK Airport,Airports
1,2,2025-10-27 13:54:05,2025-10-27 14:07:52,2,1.44,1,N,246,164,1,...,0.00,1.0,21.90,2.5,0.00,2025,10,Manhattan,West Chelsea/Hudson Yards,Yellow Zone
2,7,2025-10-01 00:00:08,2025-10-01 00:00:08,1,5.00,1,N,107,225,1,...,0.00,1.0,42.44,2.5,0.00,2025,10,Manhattan,Gramercy,Yellow Zone
3,2,2025-10-27 13:24:48,2025-10-27 13:56:18,1,9.59,1,N,239,220,2,...,3.18,1.0,47.28,2.5,0.00,2025,10,Manhattan,Upper West Side South,Yellow Zone
4,2,2025-10-01 00:08:54,2025-10-01 00:14:44,1,2.75,1,N,263,229,1,...,0.00,1.0,22.26,2.5,0.00,2025,10,Manhattan,Yorkville West,Yellow Zone
